# Human Pose Estimation: 모델 구조 및 학습 전략 비교 (Ablation Study)

## 1. 실험 목적
본 실험은 Human Pose Estimation Task에서 **모델의 구조(Architecture)**와 **학습 전략(Training Strategy)**이 성능에 미치는 영향을 분석하는 것을 목표로 합니다. 
시간적 제약을 고려하여, 가장 대조적이고 의미 있는 결과를 도출할 수 있는 3가지 실험군을 구성하였습니다.

1. **사전학습된 ResNet-50 (SimpleBaseline)**: ImageNet으로 사전 학습된 범용 백본이 Pose Estimation 태스크로 얼마나 효과적으로 전이(Transfer)되는지 확인하는 대조군입니다.
2. **4-Stack Hourglass (Deep Supervision ON)**: 공간 정보의 손실을 막기 위해 반복적인 Up/Down Sampling을 수행하는 Pose 특화 구조(Hourglass)의 성능을 검증합니다.
3. **4-Stack Hourglass (Deep Supervision OFF)**: Hourglass 구조에서 중간 감독(Deep Supervision)이 제거되었을 때 기울기 전파(Gradient Flow)와 최종 성능에 미치는 영향을 확인하는 아블레이션 그룹입니다.

In [1]:
import io
import json
import os
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# 프로젝트 경로 설정
PROJECT_PATH = os.path.abspath('mpii') 
IMAGE_PATH = os.path.join(PROJECT_PATH, 'images')
MODEL_PATH = os.path.join(PROJECT_PATH, 'models')
TRAIN_JSON = os.path.join(PROJECT_PATH, 'mpii_human_pose_v1_u12_2', 'train.json')
VALID_JSON = os.path.join(PROJECT_PATH, 'mpii_human_pose_v1_u12_2', 'validation.json')

IMAGE_SHAPE = (256, 256, 3)
HEATMAP_SIZE = (64, 64)

print('라이브러리 및 환경 설정 완료')

라이브러리 및 환경 설정 완료


## 2. 데이터 전처리 및 데이터 로더 구성
원본 MPII 데이터셋의 JSON 어노테이션에서 필요한 좌표만 추출하고, ROI Crop 및 Resize를 통해 모델에 입력 가능한 텐서 형태로 변환합니다. 정답 좌표는 모델 학습에 유리한 Heatmap (2D Gaussian) 형태로 변경됩니다.

In [2]:
def parse_one_annotation(anno, image_dir):
    filename = anno['image']
    joints = anno['joints']
    joints_visibility = anno['joints_vis']
    annotation = {
        'filename': filename,
        'filepath': os.path.join(image_dir, filename),
        'joints_visibility': joints_visibility,
        'joints': joints,
        'center': anno['center'],
        'scale' : anno['scale']
    }
    return annotation

def generate_2d_gaussian(height, width, y0, x0, visibility=2, sigma=1, scale=12):
    heatmap = torch.zeros((height, width), dtype=torch.float32)

    xmin = x0 - 3 * sigma
    ymin = y0 - 3 * sigma
    xmax = x0 + 3 * sigma
    ymax = y0 + 3 * sigma

    if xmin >= width or ymin >= height or xmax < 0 or ymax < 0 or visibility == 0:
        return heatmap

    size = int(6 * sigma + 1)
    grid_range = torch.arange(0, size, dtype=torch.float32)
    x_grid, y_grid = torch.meshgrid(grid_range, grid_range, indexing='xy')
    center_x = size // 2
    center_y = size // 2

    gaussian_patch = torch.exp(-(((x_grid - center_x)**2 + (y_grid - center_y)**2) / (sigma**2 * 2))) * scale

    patch_xmin = max(0, -xmin)
    patch_ymin = max(0, -ymin)
    patch_xmax = min(xmax, width) - xmin
    patch_ymax = min(ymax, height) - ymin

    heatmap_xmin = max(0, xmin)
    heatmap_ymin = max(0, ymin)
    heatmap_xmax = min(xmax, width)
    heatmap_ymax = min(ymax, height)

    heatmap[heatmap_ymin:heatmap_ymax, heatmap_xmin:heatmap_xmax] = \
        gaussian_patch[int(patch_ymin):int(patch_ymax), int(patch_xmin):int(patch_xmax)]

    return heatmap

class Preprocessor(object):
    def __init__(self, image_shape=(256, 256, 3), heatmap_shape=(64, 64, 16), is_train=False):
        self.is_train = is_train
        self.image_shape = image_shape
        self.heatmap_shape = heatmap_shape

    def __call__(self, example):
        features = self.parse_tfexample(example)
        image = Image.open(io.BytesIO(features['image/encoded']))

        if self.is_train:
            random_margin = torch.empty(1).uniform_(0.1, 0.3).item()
            image, keypoint_x, keypoint_y = self.crop_roi(image, features, margin=random_margin)
            image = image.resize((self.image_shape[1], self.image_shape[0]))
        else:
            image, keypoint_x, keypoint_y = self.crop_roi(image, features)
            image = image.resize((self.image_shape[1], self.image_shape[0]))

        image_np = np.array(image).astype(np.float32)
        image_np = image_np / 127.5 - 1.0
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1)

        heatmaps = self.make_heatmaps(features, keypoint_x, keypoint_y, self.heatmap_shape)

        return image_tensor, heatmaps

    def parse_tfexample(self, example):
        annotation = example['annotation']
        joints = annotation['joints']
        keypoint_x = [joint[0] for joint in joints]
        keypoint_y = [joint[1] for joint in joints]
        joints_vis = annotation.get('joints_vis', [1] * len(joints))

        features = {
            'image/encoded': self.image_to_bytes(example['image']),
            'image/object/parts/x': keypoint_x,
            'image/object/parts/y': keypoint_y,
            'image/object/parts/v': joints_vis,
            'image/object/center/x': annotation['center'][0],
            'image/object/center/y': annotation['center'][1],
            'image/object/scale': annotation['scale'],
        }
        return features

    def image_to_bytes(self, image):
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG")
        return buffer.getvalue()

    def crop_roi(self, image, features, margin=0.2):
        img_width, img_height = image.size
        keypoint_x = torch.tensor(features['image/object/parts/x'], dtype=torch.int32)
        keypoint_y = torch.tensor(features['image/object/parts/y'], dtype=torch.int32)
        body_height = features['image/object/scale'] * 200.0

        masked_keypoint_x = keypoint_x[keypoint_x > 0]
        masked_keypoint_y = keypoint_y[keypoint_y > 0]

        keypoint_xmin = int(masked_keypoint_x.min().item())
        keypoint_xmax = int(masked_keypoint_x.max().item())
        keypoint_ymin = int(masked_keypoint_y.min().item())
        keypoint_ymax = int(masked_keypoint_y.max().item())

        extra = int(body_height * margin)
        xmin = keypoint_xmin - extra
        xmax = keypoint_xmax + extra
        ymin = keypoint_ymin - extra
        ymax = keypoint_ymax + extra

        effective_xmin = max(xmin, 0)
        effective_ymin = max(ymin, 0)
        effective_xmax = min(xmax, img_width)
        effective_ymax = min(ymax, img_height)

        cropped_image = image.crop((effective_xmin, effective_ymin, effective_xmax, effective_ymax))

        new_width = effective_xmax - effective_xmin
        new_height = effective_ymax - effective_ymin

        effective_keypoint_x = (keypoint_x.float() - effective_xmin) / new_width
        effective_keypoint_y = (keypoint_y.float() - effective_ymin) / new_height

        return cropped_image, effective_keypoint_x, effective_keypoint_y

    def make_heatmaps(self, features, keypoint_x, keypoint_y, heatmap_shape):
        v = torch.tensor(features['image/object/parts/v'], dtype=torch.float32)
        x = torch.round(keypoint_x * heatmap_shape[1]).to(torch.int32)
        y = torch.round(keypoint_y * heatmap_shape[0]).to(torch.int32)

        num_heatmap = heatmap_shape[2]
        heatmaps_list = []

        for i in range(num_heatmap):
            gaussian = generate_2d_gaussian(
                heatmap_shape[0], heatmap_shape[1],
                int(y[i].item()), int(x[i].item()), visibility=int(v[i].item())
            )
            heatmaps_list.append(gaussian)

        heatmaps = torch.stack(heatmaps_list, dim=0)
        return heatmaps

class MPIIDataset(Dataset):
    def __init__(self, annotation_file, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        with open(annotation_file, 'r') as f:
            annotations = json.load(f)
        self.annotations = [parse_one_annotation(anno, image_dir) for anno in annotations]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        anno = self.annotations[idx]
        image = Image.open(anno['filepath']).convert('RGB')
        if self.transform:
            image, heatmaps = self.transform({'image': image, 'annotation': anno})
            return image, heatmaps
        return image, anno

def create_dataloader(annotation_file, image_dir, batch_size, num_heatmap, is_train=True):
    preprocess = Preprocessor(
        image_shape=IMAGE_SHAPE,
        heatmap_shape=(HEATMAP_SIZE[0], HEATMAP_SIZE[1], num_heatmap),
        is_train=is_train
    )
    dataset = MPIIDataset(annotation_file=annotation_file, image_dir=image_dir, transform=preprocess)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=is_train,
        num_workers=4, pin_memory=True, drop_last=False, prefetch_factor=2
    )
    return dataloader

## 3. 모델 설계

### 3.1 SimpleBaseline (Pretrained ResNet-50)
기존 분류용 모델인 ResNet-50의 마지막 레이어를 제거하고, 3개의 Deconvolution Layer를 추가하여 해상도를 복원(Upsampling)하는 직관적이고 심플한 구조입니다. 
미리 학습된 가중치(Pretrained weights)를 사용하기 때문에 수렴이 빠르고 안정적인 성능을 보일 것으로 기대됩니다.

In [4]:
class SimpleBaseline(nn.Module):
    def __init__(self, num_heatmap=16):
        super(SimpleBaseline, self).__init__()
        
        # 1. Pretrained ResNet-50 불러오기
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        # FC Layer와 AvgPool 제거 (Features 맵만 추출)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        
        # 2. Deconvolution Layer 생성 (3개의 ConvTranspose2d)
        # ResNet-50의 출력 채널은 2048
        self.deconv_layers = self._make_deconv_layer(
            num_layers=3,
            num_filters=[256, 256, 256],
            num_kernels=[4, 4, 4]
        )
        
        # 3. Final Conv Layer (Heatmap 생성)
        self.final_layer = nn.Conv2d(
            in_channels=256,
            out_channels=num_heatmap,
            kernel_size=1,
            stride=1,
            padding=0
        )

    def _make_deconv_layer(self, num_layers, num_filters, num_kernels):
        layers = []
        in_channels = 2048
        for i in range(num_layers):
            kernel, padding, output_padding = num_kernels[i], 1, 0
            planes = num_filters[i]
            layers.append(
                nn.ConvTranspose2d(
                    in_channels=in_channels,
                    out_channels=planes,
                    kernel_size=kernel,
                    stride=2,
                    padding=padding,
                    output_padding=output_padding,
                    bias=False))
            layers.append(nn.BatchNorm2d(planes, momentum=0.1))
            layers.append(nn.ReLU(inplace=True))
            in_channels = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.backbone(x)
        x = self.deconv_layers(x)
        x = self.final_layer(x)
        
        # Trainer 구조 호환을 위해 리스트 형태로 반환 (단일 출력)
        return [x]

### 3.2 Stacked Hourglass Network
Bottleneck 블록과 모래시계(Hourglass) 모듈을 겹겹이 쌓아 올린 특화 구조입니다. 
`deep_supervision` 매개변수를 추가하여, 훈련 시 매 스택마다 Loss를 계산할 것인지(ON), 아니면 마지막 스택에서만 계산할 것인지(OFF) 제어할 수 있도록 설계하였습니다.

In [5]:
class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, filters, stride=1, downsample=False):
        super(BottleneckBlock, self).__init__()
        self.downsample = downsample
        if self.downsample:
            self.downsample_conv = nn.Conv2d(in_channels, filters, kernel_size=1, stride=stride, bias=False)

        self.bn1 = nn.BatchNorm2d(in_channels, momentum=0.9)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, filters // 2, kernel_size=1, stride=1, padding=0, bias=False)

        self.bn2 = nn.BatchNorm2d(filters // 2, momentum=0.9)
        self.conv2 = nn.Conv2d(filters // 2, filters // 2, kernel_size=3, stride=stride, padding=1, bias=False)

        self.bn3 = nn.BatchNorm2d(filters // 2, momentum=0.9)
        self.conv3 = nn.Conv2d(filters // 2, filters, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        identity = x
        if self.downsample:
            identity = self.downsample_conv(x)

        out = self.bn1(x)
        out = self.relu(out)
        out = self.conv1(out)

        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv2(out)

        out = self.bn3(out)
        out = self.relu(out)
        out = self.conv3(out)

        out += identity
        return out

class HourglassModule(nn.Module):
    def __init__(self, order, filters, num_residual):
        super(HourglassModule, self).__init__()
        self.order = order

        self.up1_0 = BottleneckBlock(in_channels=filters, filters=filters, stride=1, downsample=False)
        self.up1_blocks = nn.Sequential(*[
            BottleneckBlock(in_channels=filters, filters=filters, stride=1, downsample=False)
            for _ in range(num_residual)
        ])

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.low1_blocks = nn.Sequential(*[
            BottleneckBlock(in_channels=filters, filters=filters, stride=1, downsample=False)
            for _ in range(num_residual)
        ])

        if order > 1:
            self.low2 = HourglassModule(order - 1, filters, num_residual)
        else:
            self.low2_blocks = nn.Sequential(*[
                BottleneckBlock(in_channels=filters, filters=filters, stride=1, downsample=False)
                for _ in range(num_residual)
            ])

        self.low3_blocks = nn.Sequential(*[
            BottleneckBlock(in_channels=filters, filters=filters, stride=1, downsample=False)
            for _ in range(num_residual)
        ])

        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, x):
        up1 = self.up1_0(x)
        up1 = self.up1_blocks(up1)

        low1 = self.pool(x)
        low1 = self.low1_blocks(low1)
        if self.order > 1:
            low2 = self.low2(low1)
        else:
            low2 = self.low2_blocks(low1)
        low3 = self.low3_blocks(low2)
        up2 = self.upsample(low3)

        return up2 + up1

class LinearLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(LinearLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn = nn.BatchNorm2d(out_channels, momentum=0.9)
        self.relu = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class StackedHourglassNetwork(nn.Module):
    def __init__(self, input_shape=(256, 256, 3), num_stack=4, num_residual=1, num_heatmap=16, deep_supervision=True):
        super(StackedHourglassNetwork, self).__init__()
        self.num_stack = num_stack
        self.deep_supervision = deep_supervision

        in_channels = input_shape[2]
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64, momentum=0.9)
        self.relu = nn.ReLU(inplace=True)

        self.bottleneck1 = BottleneckBlock(in_channels=64, filters=128, stride=1, downsample=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck2 = BottleneckBlock(in_channels=128, filters=128, stride=1, downsample=False)
        self.bottleneck3 = BottleneckBlock(in_channels=128, filters=256, stride=1, downsample=True)

        self.hourglass_modules = nn.ModuleList()
        self.residual_modules = nn.ModuleList()
        self.linear_layers = nn.ModuleList()
        self.heatmap_convs = nn.ModuleList()
        self.intermediate_convs = nn.ModuleList()
        self.intermediate_outs = nn.ModuleList()

        for i in range(num_stack):
            self.hourglass_modules.append(HourglassModule(order=4, filters=256, num_residual=num_residual))
            self.residual_modules.append(nn.Sequential(*[
                BottleneckBlock(in_channels=256, filters=256, stride=1, downsample=False)
                for _ in range(num_residual)
            ]))
            self.linear_layers.append(LinearLayer(in_channels=256, out_channels=256))
            self.heatmap_convs.append(nn.Conv2d(256, num_heatmap, kernel_size=1, stride=1, padding=0))

            if i < num_stack - 1:
                self.intermediate_convs.append(nn.Conv2d(256, 256, kernel_size=1, stride=1, padding=0))
                self.intermediate_outs.append(nn.Conv2d(num_heatmap, 256, kernel_size=1, stride=1, padding=0))

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.bottleneck1(x)
        x = self.pool(x)
        x = self.bottleneck2(x)
        x = self.bottleneck3(x)

        outputs = []
        for i in range(self.num_stack):
            hg = self.hourglass_modules[i](x)
            res = self.residual_modules[i](hg)
            lin = self.linear_layers[i](res)
            heatmap = self.heatmap_convs[i](lin)
            
            # Deep Supervision 설정에 따른 반환 리스트 구성
            if self.deep_supervision:
                outputs.append(heatmap)
            elif i == self.num_stack - 1:
                outputs.append(heatmap) # OFF인 경우 마지막 스택만 추가

            if i < self.num_stack - 1:
                inter1 = self.intermediate_convs[i](lin)
                inter2 = self.intermediate_outs[i](heatmap)
                x = inter1 + inter2

        return outputs

## 4. 학습 엔진 구현
세 가지 모델의 반환 형태(단일 Heatmap 리스트 vs 다중 Heatmap 리스트)를 유연하게 처리할 수 있는 범용 Trainer 클래스와 통합 학습 함수를 구현합니다. `compute_loss` 메서드는 리스트에 담긴 출력 수량에 관계없이 순회하며 오차를 누적하므로 모든 실험군에 정상 동작합니다.

In [ ]:
import os
import time
import math
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

class Trainer(object):
    def __init__(self, model, epochs, global_batch_size, initial_learning_rate):
        self.model = model
        self.epochs = epochs
        self.global_batch_size = global_batch_size
        self.loss_object = nn.MSELoss(reduction='none')
        self.optimizer = optim.Adam(self.model.parameters(), lr=initial_learning_rate)

        self.current_learning_rate = initial_learning_rate
        self.last_val_loss = math.inf
        self.lowest_val_loss = math.inf
        self.patience_count = 0
        self.max_patience = 10
        self.best_model = None

        if torch.cuda.device_count() > 1:
            self.model = nn.DataParallel(self.model)

    def lr_decay(self):
        if self.patience_count >= self.max_patience:
            self.current_learning_rate /= 10.0
            self.patience_count = 0
        elif self.last_val_loss == self.lowest_val_loss:
            self.patience_count = 0

        self.patience_count += 1
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self.current_learning_rate

    def compute_loss(self, labels, outputs):
        loss = 0
        for output in outputs:
            weights = (labels > 0).float() * 81 + 1
            squared_error = (labels - output) ** 2
            weighted_error = squared_error * weights
            loss += weighted_error.mean() / self.global_batch_size
        return loss

    def train_step(self, images, labels, device):
        self.model.train()
        images = images.to(device)
        labels = labels.to(device)

        self.optimizer.zero_grad()
        outputs = self.model(images)
        loss = self.compute_loss(labels, outputs)
        loss.backward()
        self.optimizer.step()

        return loss.item()

    def val_step(self, images, labels, device):
        self.model.eval()
        with torch.no_grad():
            images = images.to(device)
            labels = labels.to(device)
            outputs = self.model(images)
            loss = self.compute_loss(labels, outputs)
        return loss.item()

    def run(self, train_loader, val_loader, device, model_name_prefix):
        # 결과를 담을 바구니(리스트) 생성
        history = []

        for epoch in range(1, self.epochs + 1):
            epoch_start_time = time.time()  # 에폭 시작 시간 기록
            self.lr_decay()
            print(f"\n[{model_name_prefix}] Start epoch {epoch} with LR {self.current_learning_rate:.6f}")

            # ================= Training =================
            total_train_loss = 0.0
            num_train_batches = 0
            
            # tqdm을 사용한 진행바 래핑
            pbar_train = tqdm(train_loader, desc=f"Epoch {epoch} [Train]", leave=False)
            for images, labels in pbar_train:
                batch_loss = self.train_step(images, labels, device)
                total_train_loss += batch_loss
                num_train_batches += 1
                
                # 진행바 우측에 현재 loss와 평균 loss 표시
                pbar_train.set_postfix({'loss': f"{batch_loss:.4f}", 'avg_loss': f"{total_train_loss/num_train_batches:.4f}"})
                
            train_loss = total_train_loss / num_train_batches

            # ================= Validation =================
            total_val_loss = 0.0
            num_val_batches = 0
            
            pbar_val = tqdm(val_loader, desc=f"Epoch {epoch} [Val]", leave=False)
            for images, labels in pbar_val:
                batch_loss = self.val_step(images, labels, device)
                num_val_batches += 1
                if not math.isnan(batch_loss):
                    total_val_loss += batch_loss
                else:
                    num_val_batches -= 1
                    
                pbar_val.set_postfix({'val_loss': f"{batch_loss:.4f}"})

            if num_val_batches > 0:
                val_loss = total_val_loss / num_val_batches
            else:
                val_loss = float('nan')

            # ================= 에폭 결과 정리 =================
            epoch_time = time.time() - epoch_start_time  # 에폭 소요 시간 계산
            print(f"[{model_name_prefix}] Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.2f}s")

            # 바구니에 현재 에폭의 정보 담기
            history.append({
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'time_seconds': epoch_time
            })

            # 최저 val_loss 갱신 시 모델 저장
            if val_loss < self.lowest_val_loss:
                self.save_model(epoch, val_loss, model_name_prefix)
                self.lowest_val_loss = val_loss
            self.last_val_loss = val_loss

        # ================= 모든 학습 종료 후 CSV 저장 =================
        history_df = pd.DataFrame(history)
        csv_path = os.path.join(MODEL_PATH, f'{model_name_prefix}_history.csv')
        history_df.to_csv(csv_path, index=False)
        print(f"\n[완료] 학습 기록이 {csv_path} 에 성공적으로 저장되었습니다.")

        return self.best_model

    def save_model(self, epoch, loss, prefix):
        model_name = os.path.join(MODEL_PATH, f'{prefix}-epoch-{epoch}-loss-{loss:.4f}.pt')
        torch.save(self.model.state_dict(), model_name)
        self.best_model = model_name
        print(f"  -> Model {model_name} saved.")


def train_experiment(model_type, epochs, learning_rate, num_heatmap, batch_size, train_json, val_json, image_dir):
    print(f"========== Start Experiment: {model_type} ==========")
    
    train_loader = create_dataloader(train_json, image_dir, batch_size, num_heatmap, is_train=True)
    val_loader = create_dataloader(val_json, image_dir, batch_size, num_heatmap, is_train=False)

    if not os.path.exists(MODEL_PATH):
        os.makedirs(MODEL_PATH)

    # 지정된 model_type에 따라 적절한 모델 인스턴스화
    if model_type == 'SimpleBaseline':
        model = SimpleBaseline(num_heatmap=num_heatmap)
    elif model_type == 'Hourglass_DS_ON':
        model = StackedHourglassNetwork(IMAGE_SHAPE, num_stack=4, num_residual=1, num_heatmap=num_heatmap, deep_supervision=True)
    elif model_type == 'Hourglass_DS_OFF':
        model = StackedHourglassNetwork(IMAGE_SHAPE, num_stack=4, num_residual=1, num_heatmap=num_heatmap, deep_supervision=False)
    else:
        raise ValueError("알 수 없는 모델 타입입니다.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    trainer = Trainer(
        model,
        epochs,
        batch_size,
        initial_learning_rate=learning_rate
    )

    return trainer.run(train_loader, val_loader, device, model_name_prefix=model_type)


## 5. 실험 진행
준비된 파이프라인을 바탕으로 3가지 모델 구성을 각각 학습시킵니다. 데이터 로딩과 학습 환경에 따라 다소 시간이 소요될 수 있습니다.

In [ ]:
# 학습 파라미터 공통 설정
EPOCHS = 10
BATCH_SIZE = 16
NUM_HEATMAP = 16
LEARNING_RATE = 0.0007

# 1. SimpleBaseline (ResNet-50) 학습
# best_model_simple = train_experiment(
#     'SimpleBaseline', EPOCHS, LEARNING_RATE, NUM_HEATMAP, BATCH_SIZE, 
#     TRAIN_JSON, VALID_JSON, IMAGE_PATH
# )

# 2. 4-Stack Hourglass (Deep Supervision ON) 학습
# best_model_hg_ds_on = train_experiment(
#     'Hourglass_DS_ON', EPOCHS, LEARNING_RATE, NUM_HEATMAP, BATCH_SIZE, 
#     TRAIN_JSON, VALID_JSON, IMAGE_PATH
# )

# 3. 4-Stack Hourglass (Deep Supervision OFF) 학습
# best_model_hg_ds_off = train_experiment(
#     'Hourglass_DS_OFF', EPOCHS, LEARNING_RATE, NUM_HEATMAP, BATCH_SIZE, 
#     TRAIN_JSON, VALID_JSON, IMAGE_PATH
# )

print("실험 코드 구성이 완료되었습니다. 주석을 해제하여 학습을 진행할 수 있습니다.")